# Gradient Boosting + MLP 5-Fold


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

DATA=Path("data")
MODELS=Path("models")
MLP_DIR=MODELS/"medium_5fold"
GB_DIR=MODELS/"gb_5fold"
GB_DIR.mkdir(parents=True,exist_ok=True)

TRAIN_PATH=DATA/"train.csv"
TEST_PATH=DATA/"test_features-1.csv"
EXPECTED_PATH=Path("expected_output.csv")

SEED=42
N_SPLITS=5


In [4]:
train_df=pd.read_csv(TRAIN_PATH)
test_df=pd.read_csv(TEST_PATH)

X=train_df.drop(columns=["SalePrice","Id"]).copy()
y=train_df["SalePrice"].to_numpy(dtype=float)
Xtest=test_df.drop(columns=["Id","SalePrice"],errors="ignore").copy()
test_ids=test_df["Id"].copy()

y_bins=pd.qcut(pd.Series(y),q=10,labels=False,duplicates="drop").to_numpy()
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
splits=list(skf.split(X,y_bins))

print(X.shape,Xtest.shape)


(1168, 79) (292, 79)


## Recuperar OOF y predicciones test de la MLP 5-Fold


In [ ]:
mlp_oof_path=MLP_DIR/"oof_predictions.npy"
if not mlp_oof_path.exists():
    raise FileNotFoundError("Falta models/medium_5fold/oof_predictions.npy. Ejecuta primero el notebook 11.")

mlp_oof=np.load(mlp_oof_path)

mlp_test_path=MLP_DIR/"test_predictions.npy"

if mlp_test_path.exists():
    mlp_test=np.load(mlp_test_path)
else:
    import torch
    import torch.nn as nn

    DEVICE="cuda" if torch.cuda.is_available() else "cpu"

    class MLP(nn.Module):
        def __init__(self,input_dim,hidden,dropout=0.0):
            super().__init__()
            layers=[]; prev=input_dim
            for h in hidden:
                layers += [nn.Linear(prev,h),nn.ReLU()]
                if dropout>0: layers.append(nn.Dropout(dropout))
                prev=h
            layers.append(nn.Linear(prev,1))
            self.net=nn.Sequential(*layers)
        def forward(self,x): return self.net(x).squeeze(1)

    fold_test=[]
    for fold in range(1,6):
        bundle=joblib.load(MLP_DIR/f"fold_{fold}_preprocessor.joblib")
        ckpt=torch.load(MLP_DIR/f"fold_{fold}_model.pt",map_location=DEVICE)

        prep=bundle["preprocessor"]
        Xte=prep.transform(Xtest).astype(np.float32)

        hp=ckpt["hyperparams"]
        model=MLP(Xte.shape[1],ckpt["hidden_layers"],hp["dropout"]).to(DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()

        with torch.no_grad():
            ps=model(torch.tensor(Xte,dtype=torch.float32,device=DEVICE)).cpu().numpy()

        pred=ps*float(ckpt["y_std"])+float(ckpt["y_mean"])
        fold_test.append(pred)

    mlp_test=np.mean(np.vstack(fold_test),axis=0)
    np.save(mlp_test_path,mlp_test)

rmse_mlp=np.sqrt(np.mean((mlp_oof-y)**2))
print(f"RMSE OOF MLP: ${rmse_mlp:,.2f}")


RMSE OOF MLP: $33,877.00


## Gradient Boosting 5-Fold


In [6]:
def make_preprocessor(Xtr):
    nums=Xtr.select_dtypes(include=[np.number]).columns.tolist()
    cats=Xtr.select_dtypes(exclude=[np.number]).columns.tolist()

    num=Pipeline([
        ("imputer",SimpleImputer(strategy="median",add_indicator=True)),
    ])
    cat=Pipeline([
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
    ])
    return ColumnTransformer([("num",num,nums),("cat",cat,cats)])

configs=[
    dict(n_estimators=500,learning_rate=0.03,max_depth=3,min_samples_leaf=5,subsample=0.8,max_features=None),
    dict(n_estimators=700,learning_rate=0.02,max_depth=3,min_samples_leaf=5,subsample=0.8,max_features="sqrt"),
]

config_results=[]
all_config_oof=[]
all_config_test=[]

for ci,cfg in enumerate(configs):
    print("\nCONFIG",ci+1,cfg)
    oof=np.zeros(len(X))
    fold_test=[]

    for fold,(tri,vai) in enumerate(splits,1):
        prep=make_preprocessor(X.iloc[tri])
        Xtr=prep.fit_transform(X.iloc[tri])
        Xva=prep.transform(X.iloc[vai])
        Xte=prep.transform(Xtest)

        model=GradientBoostingRegressor(random_state=SEED+fold,**cfg)
        model.fit(Xtr,y[tri])

        oof[vai]=model.predict(Xva)
        fold_test.append(model.predict(Xte))

        print(f" fold {fold}: RMSE ${np.sqrt(np.mean((oof[vai]-y[vai])**2)):,.2f}")

    testpred=np.mean(np.vstack(fold_test),axis=0)
    rmse=np.sqrt(np.mean((oof-y)**2))
    print(f" OOF config {ci+1}: ${rmse:,.2f}")

    config_results.append({"config":ci+1,"RMSE_OOF":rmse,**cfg})
    all_config_oof.append(oof)
    all_config_test.append(testpred)

config_df=pd.DataFrame(config_results).sort_values("RMSE_OOF")
display(config_df)



CONFIG 1 {'n_estimators': 500, 'learning_rate': 0.03, 'max_depth': 3, 'min_samples_leaf': 5, 'subsample': 0.8, 'max_features': None}
 fold 1: RMSE $23,314.82
 fold 2: RMSE $33,116.99
 fold 3: RMSE $23,397.89
 fold 4: RMSE $25,093.42
 fold 5: RMSE $50,894.88
 OOF config 1: $32,873.15

CONFIG 2 {'n_estimators': 700, 'learning_rate': 0.02, 'max_depth': 3, 'min_samples_leaf': 5, 'subsample': 0.8, 'max_features': 'sqrt'}
 fold 1: RMSE $23,163.31
 fold 2: RMSE $32,335.89
 fold 3: RMSE $24,911.05
 fold 4: RMSE $26,452.45
 fold 5: RMSE $47,030.42
 OOF config 2: $31,970.59


,config,RMSE_OOF,n_estimators,learning_rate,max_depth,min_samples_leaf,subsample,max_features
1,2,31970.593003,700,0.02,3,5,0.8,sqrt
0,1,32873.153319,500,0.03,3,5,0.8,None


In [7]:
best_ci=int(config_df.iloc[0]["config"])-1
gb_oof=all_config_oof[best_ci]
gb_test=all_config_test[best_ci]

rmse_gb=np.sqrt(np.mean((gb_oof-y)**2))
print(f"RMSE OOF GB elegido: ${rmse_gb:,.2f}")


RMSE OOF GB elegido: $31,970.59


## Optimizar blend 


In [8]:
rows=[]
for w in np.linspace(0,1,101):
    pred=w*mlp_oof+(1-w)*gb_oof
    rmse=np.sqrt(np.mean((pred-y)**2))
    rows.append({"peso_MLP":w,"peso_GB":1-w,"RMSE_OOF":rmse})

blend_df=pd.DataFrame(rows).sort_values("RMSE_OOF").reset_index(drop=True)
display(blend_df.head(15))

best=blend_df.iloc[0]
w=float(best["peso_MLP"])
blend_oof=w*mlp_oof+(1-w)*gb_oof
blend_test=w*mlp_test+(1-w)*gb_test

print(f"RMSE OOF MLP:   ${rmse_mlp:,.2f}")
print(f"RMSE OOF GB:    ${rmse_gb:,.2f}")
print(f"Peso MLP:       {w:.2f}")
print(f"Peso GB:        {1-w:.2f}")
print(f"RMSE OOF Blend: ${best['RMSE_OOF']:,.2f}")
print(f"Mejora vs MLP:  ${rmse_mlp-best['RMSE_OOF']:,.2f}")


,peso_MLP,peso_GB,RMSE_OOF
0,0.30,0.70,31519.510244
1,0.31,0.69,31519.891866
2,0.29,0.71,31520.130246
3,0.32,0.68,31521.275076
4,0.28,0.72,31521.751814
5,0.33,0.67,31523.659743
6,0.27,0.73,31524.374792
7,0.34,0.66,31527.045638
8,0.26,0.74,31527.998932
9,0.35,0.65,31531.432440


RMSE OOF MLP:   $33,877.00
RMSE OOF GB:    $31,970.59
Peso MLP:       0.30
Peso GB:        0.70
RMSE OOF Blend: $31,519.51
Mejora vs MLP:  $2,357.49


## Generar csv


In [9]:
out=pd.DataFrame({"Id":test_ids.to_numpy(),"Prediction":blend_test})
assert out.columns.tolist()==["Id","Prediction"]
assert out["Prediction"].notna().all()

if EXPECTED_PATH.exists():
    exp=pd.read_csv(EXPECTED_PATH)
    assert exp.columns.tolist()==["Id","Prediction"]
    assert len(exp)==len(out)
    assert np.array_equal(exp["Id"].to_numpy(),out["Id"].to_numpy())

out.to_csv("predictions_boost.csv",index=False)

print("✓ predictions_boost.csv generado")
display(out.head())


✓ predictions_boost.csv generado


,Id,Prediction
0,893,141536.667106
1,1106,332499.119962
2,414,106798.090594
3,523,147820.921723
4,1037,328934.887625
